# Cell Candidate Probe mse Degradation

In [ ]:
from pathlib import Path
import os, sys

# Allow running from repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


## Experiment
Train one candidate probe per cell at the initial board state and evaluate AUC/mse as the board advances, split into simple vs backtracking traces.


# mse Score Degradation Analysis

**Hypothesis**: Probes trained on the `[clues_end]` token maintain AUC when evaluated at later positions, but mse score degrades. The degradation pattern differs between *simple* puzzles (no backtracking) and *hard* puzzles (with PUSH/POP tokens).

**Setup**: Candidate-set probes (9-class binary per cell) trained at step 0, evaluated across positions grouped by **number of empty cells** (`n_empty = 81 − n_filled`). This is a board-state–based axis rather than a sequence-position–based axis, making simple and hard puzzles directly comparable.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from sudoku.probes.session import ProbeSession
from sudoku.probes.modes import cell_candidates
from sudoku.probes.activations import build_grid_at_step
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN, END_CLUES_TOKEN, PAD_TOKEN_BT

In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
LAYER = 4     # layer to probe (0-indexed)
N_CELLS = 81    # probe all cells
SEED = 42

## Load session

In [ ]:
session = ProbeSession(CACHE_PATH)

## Classify puzzles: simple vs hard

Simple = no PUSH/POP tokens anywhere in the sequence (pure constraint propagation).  
Hard = at least one PUSH token (backtracking was used).

In [ ]:
is_simple = np.array([
    not any(t == PUSH_TOKEN for t in seq)
    for seq in session.sequences
])

n_simple = is_simple.sum()
n_hard = (~is_simple).sum()
print(f"Simple puzzles (no backtracking): {n_simple:,}")
print(f"Hard puzzles (with backtracking): {n_hard:,}")

# Sequence length distribution check
seq_lens = np.array([len(s) for s in session.sequences])
print(f"\nSimple seq len: min={seq_lens[is_simple].min()}, max={seq_lens[is_simple].max()}, mean={seq_lens[is_simple].mean():.1f}")
print(f"Hard   seq len: min={seq_lens[~is_simple].min()}, max={seq_lens[~is_simple].max()}, mean={seq_lens[~is_simple].mean():.1f}")

## Build train/test split at step 0

- **Train**: all puzzles at step 0 (80% split)
- **Test**: the remaining 20%, further divided into simple/hard subsets

In [ ]:
idx0 = session.index.at_step(1).first_per_puzzle()
train_mask, test_mask = session.split(idx0, test_size=0.2, seed=SEED)

train_idx = idx0[train_mask]
test_idx  = idx0[test_mask]

# Identify which test puzzles are simple vs hard
test_puzzle_ids = test_idx.puzzle_idx
test_is_simple = is_simple[test_puzzle_ids]

print(f"Train: {len(train_idx):,}  |  Test: {len(test_idx):,}")
print(f"Test simple: {test_is_simple.sum():,}  |  Test hard: {(~test_is_simple).sum():,}")

## Helper functions

In [ ]:
from scripts.analysis.probe_experiments import (
    build_candidate_targets,
    fit_multilabel_probe as fit_cell_probe,
    eval_multilabel_probe as eval_cell_probe,
)


## Train probes at step 0

One multi-label LR probe per cell (81 probes total), trained on the training split at step 0.

In [ ]:
# Activations at step 0 for train and test splits
X_train_all = session.acts(train_idx, layer=LAYER)  # (n_train, d_model)
X_test_all  = session.acts(test_idx, layer=LAYER)   # (n_test,  d_model)

grids_train = session.grids(train_idx)
grids_test  = session.grids(test_idx)

print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

In [ ]:
# Train one probe per cell — filter to cells that have empty instances in train
cell_probes = {}  # cell_idx -> (clf, train_empty_mask)

for cell_idx in tqdm(range(N_CELLS), desc="Training probes"):
    targets_train, is_empty_train = build_candidate_targets(grids_train, cell_idx)
    if is_empty_train.sum() < 10:
        continue  # skip cells with almost no empty instances in train
    clf = fit_cell_probe(X_train_all[is_empty_train], targets_train[is_empty_train])
    cell_probes[cell_idx] = clf

print(f"Trained probes for {len(cell_probes)} cells")

In [ ]:
# Pre-identify test puzzle IDs for quick lookup
test_puzzle_id_set = set(test_puzzle_ids.tolist())
simple_test_puzzle_ids = set(test_puzzle_ids[test_is_simple].tolist())
hard_test_puzzle_ids   = set(test_puzzle_ids[~test_is_simple].tolist())


## Evaluate by number of empty cells

Instead of step offset from `[clues_end]`, group samples by **how many cells are still empty** at the probed position (`n_empty = 81 − n_filled`).

- For simple puzzles: each fill step reduces n_empty by exactly 1, so this is a monotone reparameterisation.
- For hard (BT) puzzles: n_empty can fluctuate due to backtracking; using `first_per_puzzle()` selects the first moment the board reaches that fill count (forward pass, before any rollback).

Both groups are swept over the same n_empty axis, making the comparison fair.

In [ ]:

# n_filled in the index counts fill *tokens* seen up to a position, accounting
# for PUSH/POP stack resets.  For model-generated BT sequences a re-fill after
# rollback can push the counter above 81.  Cap the sweep at [0, 81].
all_n_empty = 81 - session.index.n_filled

# Starting n_empty: largest value among step-0 samples (= 81 - n_clues)
n_empty_max = int(all_n_empty[session.index.step == 0].max())
n_empty_values = list(range(n_empty_max, -1, -1))  # high → low, stopping at 0
print(f"Sweeping n_empty from {n_empty_max} down to 0 ({len(n_empty_values)} points)")

results = {"simple": {"auc": [], "mse": [], "n": [], "n_empty": []},
           "hard":   {"auc": [], "mse": [], "n": [], "n_empty": []}}

for n_empty in tqdm(n_empty_values, desc="Evaluating n_empty"):
    n_filled_target = 81 - n_empty
    # where_filled matches the stack-adjusted counter; first_per_puzzle picks the
    # *first* forward visit (before any rollback) for BT sequences.
    idx_slot = session.index.where_filled(n_filled_target).first_per_puzzle()
    # Filter to test puzzle IDs
    in_test = np.isin(idx_slot.puzzle_idx, list(test_puzzle_id_set))
    idx_slot = idx_slot[in_test]

    if len(idx_slot) == 0:
        for grp in results:
            results[grp]["auc"].append(float("nan"))
            results[grp]["mse"].append(float("nan"))
            results[grp]["n"].append(0)
            results[grp]["n_empty"].append(n_empty)
        continue

    slot_is_simple = np.isin(idx_slot.puzzle_idx, list(simple_test_puzzle_ids))
    masks = {"simple": slot_is_simple, "hard": ~slot_is_simple}

    X_slot = session.acts(idx_slot, layer=LAYER)
    grids_slot = session.grids(idx_slot)

    for grp, mask in masks.items():
        results[grp]["n_empty"].append(n_empty)
        if mask.sum() == 0:
            results[grp]["auc"].append(float("nan"))
            results[grp]["mse"].append(float("nan"))
            results[grp]["n"].append(0)
            continue

        X_grp = X_slot[mask]
        grids_grp = [grids_slot[i] for i in np.where(mask)[0]]

        cell_aucs, cell_mses = [], []
        for cell_idx, clf in cell_probes.items():
            targets, is_empty_mask = build_candidate_targets(grids_grp, cell_idx)
            if is_empty_mask.sum() < 5:
                continue
            auc, mse = eval_cell_probe(clf, X_grp[is_empty_mask], targets[is_empty_mask])
            if not np.isnan(auc):
                cell_aucs.append(auc)
            if not np.isnan(mse):
                cell_mses.append(mse)

        results[grp]["auc"].append(float(np.mean(cell_aucs)) if cell_aucs else float("nan"))
        results[grp]["mse"].append(float(np.mean(cell_mses)) if cell_mses else float("nan"))
        results[grp]["n"].append(int(mask.sum()))

print("Done.")

## Plot results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Candidate-set probe (layer {LAYER}) trained at step 0 — evaluated by n_empty", fontsize=13)

colors = {"simple": "steelblue", "hard": "tomato"}
labels = {"simple": "Simple (no backtracking)", "hard": "Hard (backtracking)"}

for grp in ["simple", "hard"]:
    xs     = results[grp]["n_empty"]
    aucs   = results[grp]["auc"]
    mses = results[grp]["mse"]
    ns     = results[grp]["n"]

    valid_a = [(x, a) for x, a in zip(xs, aucs) if not np.isnan(a) and results[grp]["n"][xs.index(x)] > 0]
    valid_b = [(x, b) for x, b in zip(xs, mses) if not np.isnan(b) and results[grp]["n"][xs.index(x)] > 0]

    xa, ya = zip(*valid_a) if valid_a else ([], [])
    xb, yb = zip(*valid_b) if valid_b else ([], [])

    mean_n = int(np.mean([n for n in ns if n > 0])) if any(n > 0 for n in ns) else 0
    axes[0].plot(xa, ya, color=colors[grp], linewidth=1.8, marker="o", markersize=3,
                 label=f"{labels[grp]} (n≈{mean_n})")
    axes[1].plot(xb, yb, color=colors[grp], linewidth=1.8, marker="o", markersize=3)

for ax in axes:
    ax.set_xlabel("Number of empty cells (n_empty = 81 − n_filled)")
    ax.invert_xaxis()   # left = many empty (start of solve), right = few empty (end)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Mean AUC (across cells and digits)")
axes[0].set_title("AUC vs n_empty")
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)

axes[1].set_ylabel("Mean mse score (lower = better)")
axes[1].set_title("mse Score vs n_empty")
axes[1].legend(
    handles=[plt.Line2D([0], [0], color=colors[grp], linewidth=2) for grp in ["simple", "hard"]],
    labels=[labels[grp] for grp in ["simple", "hard"]],
)

plt.tight_layout()
plt.savefig("mse_degradation_by_nempty.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved mse_degradation_by_nempty.png")

## Numerical summary

In [ ]:
print(f"{'n_empty':>7}  {'AUC_simple':>10}  {'AUC_hard':>8}  {'mse_simple':>12}  {'mse_hard':>10}  {'N_simple':>8}  {'N_hard':>6}")
print("-" * 75)
for i, n_e in enumerate(results["simple"]["n_empty"]):
    def _v(grp, key): return results[grp][key][i]
    print(f"{n_e:>7}  {_v('simple','auc'):>10.4f}  {_v('hard','auc'):>8.4f}  "
          f"{_v('simple','mse'):>12.4f}  {_v('hard','mse'):>10.4f}  "
          f"{_v('simple','n'):>8}  {_v('hard','n'):>6}")